In [ ]:
# hide
from collections.abc import Callable

import numpy as np


def train_classifier(X: np.ndarray, y: np.ndarray, l2: float = 0.01, steps: int = 300,
                     learning_rate: float = 0.01) -> Callable[[np.ndarray], np.ndarray]:
    """Train a logistic regression classifier, one of the simplest models for
    classification, and return a function that predicts the labels of new inputs.

    The classifier first standardizes each feature (to a mean of 0 and a standard deviation
    of 1). Then it predicts the probability of each label as softmax(x W + b), where the
    parameters W and b are trained to minimize the average negative log-likelihood of the
    training labels, plus a small penalty (l2 / 2) ||W||^2 that discourages large weights.
    For training, it uses Adam, a variant of gradient descent that adapts the step size of
    each parameter."""
    mean, std = X.mean(axis=0), X.std(axis=0) + 1e-8
    X = (X - mean) / std
    Y = np.eye(y.max() + 1)[y]                                 # one-hot labels
    params = [np.zeros((X.shape[1], Y.shape[1])), np.zeros(Y.shape[1])]   # W and b
    m = [np.zeros_like(p) for p in params]                    # Adam's running averages
    v = [np.zeros_like(p) for p in params]
    for t in range(1, steps + 1):
        W, b = params
        z = X @ W + b
        P = np.exp(z - z.max(axis=1, keepdims=True))
        P /= P.sum(axis=1, keepdims=True)                     # softmax: P(label | x)
        G = (P - Y) / len(X)                                  # gradient of the loss w.r.t. z
        grads = [X.T @ G + l2 * W, G.sum(axis=0)]
        for p, g, m_p, v_p in zip(params, grads, m, v):
            m_p[:] = 0.9 * m_p + 0.1 * g
            v_p[:] = 0.999 * v_p + 0.001 * g ** 2
            p -= learning_rate * (m_p / (1 - 0.9 ** t)) / (np.sqrt(v_p / (1 - 0.999 ** t)) + 1e-8)
    W, b = params
    return lambda X_new: np.argmax(((X_new - mean) / std) @ W + b, axis=1)

In [ ]:
data = np.load("../assets/freetzan-features.npz")   # features of FreeTZAN's clips
genre = data["genre"]                                # labels: 0 to 9, one per genre
train, test = data["split"] == "train", data["split"] == "test"

for name in ["waveform", "spectrogram", "log_mel"]:
    X = data[name].astype(np.float64)                # one row of features per clip
    predict = train_classifier(X[train], genre[train])   # learn from the 1000 training clips
    print(f"{name:11s} ({X.shape[1]:4d} features per clip): "
          f"train accuracy {np.mean(predict(X[train]) == genre[train]):4.0%}, "
          f"test accuracy {np.mean(predict(X[test]) == genre[test]):4.0%}")